# Ejercicios adicionales: manos y video

Este notebook contiene dos aplicaciones locales de visión artificial:

1. **Tracker de mano:** usa la cámara integrada y dibuja el esqueleto de cada mano detectada.
2. **Detección en video:** permite seleccionar un archivo de video y guarda una copia con las detecciones de YOLO.

Ambos ejercicios se ejecutan en el computador local. En el tracker de mano se debe cerrar la ventana presionando `q` o `Esc`.

In [1]:
import importlib.util
import subprocess
import sys

missing_packages = [
    package for module, package in {
        "mediapipe": "mediapipe",
        "ultralytics": "ultralytics",
    }.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        *missing_packages,
    ])
    print("Paquetes instalados:", ", ".join(missing_packages))
else:
    print("Dependencias listas. Se usará la API moderna MediaPipe Tasks.")

Dependencias listas. Se usará la API moderna MediaPipe Tasks.


## 1. Tracker de mano con cámara local

MediaPipe Hands usa un modelo de visión artificial para localizar 21 puntos de referencia por mano. Las conexiones entre esos puntos forman el esqueleto de la mano y se actualizan mientras la cámara captura nuevos cuadros.

### Orden de ejecución

1. Ejecuta la celda de instalación.
2. Ejecuta la comprobación rápida; debe mostrar `Cámara disponible: True`.
3. Ejecuta la celda del tracker. Se abrirá una ventana independiente llamada `Tracker de esqueleto de mano`.
4. Coloca la mano frente a la cámara. Para terminar, selecciona esa ventana y presiona `q` o `Esc`.

No ejecutes el video al mismo tiempo que el tracker: ambos usan recursos de cámara/video y pueden parecer bloqueados.

In [3]:
# Comprobación rápida: esta celda no abre la cámara ni queda ejecutándose.
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision

camera_check = cv2.VideoCapture(0)
camera_available = camera_check.isOpened()
camera_check.release()

print("OpenCV:", cv2.__version__)
print("MediaPipe importado correctamente:", mp.__name__)
print("HandLandmarker disponible:", hasattr(vision, "HandLandmarker"))
print("Cámara disponible:", camera_available)

OpenCV: 5.0.0
MediaPipe importado correctamente: mediapipe
HandLandmarker disponible: True
Cámara disponible: True


In [4]:
import time
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

MODEL_PATH = Path("hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
if not MODEL_PATH.exists():
    print("Descargando el modelo de manos...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

# Conexiones del esqueleto de una mano: palma, dedos y pulgar.
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20),
]

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.60,
    min_hand_presence_confidence=0.60,
    min_tracking_confidence=0.60,
)

camera = cv2.VideoCapture(0)
if not camera.isOpened():
    raise RuntimeError(
        "No se pudo abrir la cámara. Cierra otras aplicaciones que la estén usando "
        "y verifica los permisos de cámara de VS Code."
    )

with vision.HandLandmarker.create_from_options(options) as landmarker:
    start_time = time.monotonic()
    while True:
        success, frame = camera.read()
        if not success:
            print("No se pudo leer un cuadro de la cámara.")
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int((time.monotonic() - start_time) * 1000)
        hand_results = landmarker.detect_for_video(mp_image, timestamp_ms)

        for landmarks in hand_results.hand_landmarks:
            points = [
                (int(landmark.x * frame.shape[1]), int(landmark.y * frame.shape[0]))
                for landmark in landmarks
            ]
            for start, end in HAND_CONNECTIONS:
                cv2.line(frame, points[start], points[end], (255, 0, 0), 2)
            for point in points:
                cv2.circle(frame, point, 4, (0, 255, 0), -1)

        cv2.putText(
            frame,
            f"Manos detectadas: {len(hand_results.hand_landmarks)} | Presiona q o Esc para salir",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 255),
            2,
        )
        cv2.imshow("Tracker de esqueleto de mano", frame)

        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):
            break

camera.release()
cv2.destroyAllWindows()
print("Tracker de mano finalizado.")

Descargando el modelo de manos...
Tracker de mano finalizado.


## 2. Detección de objetos en un video

YOLOv8 procesará cuadro por cuadro el video seleccionado. El archivo resultante incluirá cajas, nombres de clase y confianza para cada objeto detectado.

In [5]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

from ultralytics import YOLO

OUTPUT_DIR = Path("resultados_video")
OUTPUT_DIR.mkdir(exist_ok=True)

USE_FILE_DIALOG = True
VIDEO_PATH = Path("videos/video_prueba.mp4")

if USE_FILE_DIALOG:
    root = tk.Tk()
    root.withdraw()
    selected_path = filedialog.askopenfilename(
        title="Selecciona un video",
        filetypes=[
            ("Videos", "*.mp4 *.avi *.mov *.mkv *.webm"),
            ("Todos los archivos", "*.*"),
        ],
    )
    root.destroy()
    if selected_path:
        VIDEO_PATH = Path(selected_path)

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el video: {VIDEO_PATH}. Selecciona un archivo válido o cambia USE_FILE_DIALOG."
    )

print(f"Video seleccionado: {VIDEO_PATH}")
print(f"Salida esperada: {OUTPUT_DIR.resolve()}")

FileNotFoundError: No se encontró el video: videos\video_prueba.mp4. Selecciona un archivo válido o cambia USE_FILE_DIALOG.

In [ ]:
video_model = YOLO("yolov8n.pt")

video_results = video_model.predict(
    source=str(VIDEO_PATH),
    conf=0.35,
    imgsz=640,
    device="cpu",
    save=True,
    project=str(OUTPUT_DIR),
    name="detecciones",
    exist_ok=True,
    stream=True,
    verbose=False,
)

processed_frames = 0
classes_detected = set()
for frame_result in video_results:
    processed_frames += 1
    if frame_result.boxes is not None:
        classes_detected.update(
            frame_result.names[int(class_id)]
            for class_id in frame_result.boxes.cls.cpu().numpy()
        )

output_video_dir = OUTPUT_DIR / "detecciones"
print(f"Cuadros procesados: {processed_frames}")
print("Clases detectadas:", ", ".join(sorted(classes_detected)) or "ninguna")
print(f"Video anotado guardado en: {output_video_dir.resolve()}")